# Creating a Custom Pipeline Variant

Two levels of customization:

1. **Parameters** — tune the fixed default preprocessing with keyword
   arguments, e.g. `Baseline(chunk_target_tokens=200)`.
2. **Stage override** — subclass and reimplement `preprocess` / `build_kg`.

This notebook builds one variant — `SentencePipeline` — registers it in
`PIPELINE_REGISTRY`, and runs it end-to-end. Everything is **offline-safe**
(sentence chunking + regex / ontology-rules extraction).

In [ ]:
from kglab.kg_build import build_kg_into, extract, resolve
from kglab.kg_build.build import NetworkXGraphWriter
from kglab.pipelines import PIPELINE_REGISTRY, Baseline
from kglab.preprocess import chunk, clean, load, quality

In [ ]:
# ── Level 1: tune parameters with keywords ─────────────────────
small = Baseline(chunk_target_tokens=200)
print(type(small).__name__, "created — chunk_target_tokens=200")


# ── Level 2: override stages — the only way to change the architecture ──
class SentencePipeline(Baseline):
    """Sentence chunking + regex extraction (offline-safe)."""

    def preprocess(self):
        docs = clean.normalize(load.from_paths(self.input_paths))
        chunks = chunk.by_sentence(docs, target_tokens=200, overlap_tokens=60)
        return quality.filter(chunks)

    def build_kg(self, chunks):
        ontology = self._load_ontology()
        entities, triples = extract.with_methods(
            chunks, ontology, entity_method="regex", relation_method="ontology_rules"
        )
        resolved, id_map = resolve.with_method_and_mapping(entities)
        triples = [(id_map.get(s, s), p, id_map.get(o, o), *rest) for (s, p, o, *rest) in triples]
        writer = NetworkXGraphWriter(ontology=ontology)
        build_kg_into(writer, chunks, resolved, triples)
        return {"graph": writer.graph, "entities": resolved, "triples": triples}

In [ ]:
# ── Register the variant so it is discoverable by name ─────────
PIPELINE_REGISTRY["sentence"] = SentencePipeline
print("registered variants:", sorted(PIPELINE_REGISTRY))
# → discoverable by the CLI (`python main.py --variant sentence`) and by
#   BenchmarkRunner YAML configs (`variant: sentence`).

In [ ]:
# ── Run the variant end-to-end on real documents ───────────────
kg = SentencePipeline().execute(
    input_paths=["data/wikipedia/connected.jsonl"],
    output_dir="output/tutorial_custom_pipeline",
)
print(f"KG: {kg['graph'].number_of_nodes()} nodes, {kg['graph'].number_of_edges()} edges")

## Recap & where to go next

- **Parameters** — `Baseline(chunk_target_tokens=200)` tunes the default
  preprocessing.
- **Stage override** — subclass and reimplement `preprocess` / `build_kg` to
  change the architecture; keep the build storage-agnostic via `build_kg_into`
  + a `GraphWriter`.
- **Registering** in `PIPELINE_REGISTRY` makes a variant discoverable by the
  CLI (`python main.py --variant <name>`) and by `BenchmarkRunner` YAML configs.

Next: [benchmarking.ipynb](benchmarking.ipynb) to score variants against the
bundled gold; [kg_storage_agnostic.ipynb](kg_storage_agnostic.ipynb) for
storage backends.